In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.metrics import compute_metrics
from tsfm_lens.utils import (
    apply_custom_style,
    clear_cuda_cache,
    load_dyst_samples,
    set_seed,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
DEFAULT_CONTEXT_LENGTH = 512

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
clear_cuda_cache(device)
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "ablations_chronos", "examples")
os.makedirs(plot_save_dir, exist_ok=True)


### Load Pipeline

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads
print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
pipeline.__class__.__name__

### Load Data

In [ ]:
split_name = "gift-eval"
# split_name = "base40"

subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Thomas"
    system_name_pretty = system_name
    term = ""

    context_length = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length

    sample_idx = 0
    selected_dim = [0]

    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :])
    print(dyst_coords.shape)
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context = dyst_coords[:, context_start_time:context_end_time]
    print(context.shape)

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
    print(f"future_vals shape: {future_vals.shape}")

elif split_name == "gift-eval":
    from itertools import islice

    split_dir = os.path.join(DATA_DIR, split_name)
    # system_name = "m4_monthly"
    # system_name = "solar/D"
    # system_name = "SZ_TAXI/15T"
    system_name = "ett2/W"
    to_univariate, term = False, "medium"
    selected_sample_idx, selected_dim = 0, [0]

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)

    # pretty name
    system_name_pretty = f"{system_name.replace('/', '_')}_dim{selected_dim}_sample{selected_sample_idx}"

    # prediction_length = dataset.test_data.prediction_length
    prediction_length = 512
    if prediction_length != dataset.test_data.prediction_length:
        print(f"NOTE: prediction length mismatch: {prediction_length} != {dataset.test_data.prediction_length}")
        print("Make sure this is an intentional user choice")

    print(f"Prediction length: {prediction_length}, Windows: {dataset.windows}")

    # Get the selected sample efficiently
    context_entry = next(islice(dataset.test_data.input, selected_sample_idx, None))
    future_entry = next(islice(dataset.test_data.label, selected_sample_idx, None))
    print(f"Item: {context_entry['item_id']}, Freq: {context_entry['freq']}")
    print(f"context shape: {context_entry['target'].shape}")

    # Extract and select dimensions
    context_target = context_entry["target"]
    future_target = future_entry["target"]
    if context_target.ndim == 1:
        context_target, future_target = context_target[None, :], future_target[None, :]
    else:
        context_target, future_target = context_target[selected_dim], future_target[selected_dim]

    # Limit context to last DEFAULT_CONTEXT_LENGTH points and adjust start time
    orig_context_len = context_target.shape[1]
    truncate_offset = max(0, orig_context_len - DEFAULT_CONTEXT_LENGTH)
    context_target = context_target[:, -DEFAULT_CONTEXT_LENGTH:]
    context_start = context_entry["start"] + truncate_offset
    context_timesteps = np.arange(orig_context_len)[-DEFAULT_CONTEXT_LENGTH:]
    future_timesteps = np.arange(orig_context_len, orig_context_len + prediction_length)
    print(f"Context: {context_target.shape}, Future: {future_target.shape}")

    # Plot
    num_variates = context_target.shape[0]
    fig, axes = plt.subplots(num_variates, 1, figsize=(5, 2 * num_variates), squeeze=False)
    for dim, ax in enumerate(axes.flat):
        to_pandas({"target": context_target[dim], "start": context_start}).plot(
            color="black", linewidth=1, ax=ax, label="Context"
        )
        to_pandas({"target": future_target[dim], "start": future_entry["start"]}).plot(
            color="tab:blue", linewidth=1, ax=ax, label="Ground Truth"
        )
        ax.grid(which="both")
        ax.set_title(f"Dim {dim}")
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Convert to torch tensors
    context = torch.tensor(context_target)
    future_vals = torch.tensor(future_target)
    num_nans = np.isnan(context_target).sum() + np.isnan(future_target).sum()
    print(f"Context: {context.shape}, Future: {future_vals.shape}, NaNs: {num_nans}")

### Ablations

In [ ]:
pipeline.remove_all_hooks()

# NOTE: this is the top 30% of highest entropy heads (excluding all the heads in layer 0). Note that it preserves the parroting
# Specifically, these heads were identified from a context-parroting window of Lorenz system (see notebooks/kernel_analogy/head_sharpness.ipynb)
heads_to_ablate = [
    (0, 4),  # layer 0
    (10, 11),
    (9, 5),
    (10, 5),
    (0, 11),  # layer 0
    (5, 8),
    (8, 6),
    (10, 8),
    (5, 2),
    (10, 1),
    (10, 4),
    (7, 2),
    (10, 10),
    (9, 1),
    (0, 10),  # layer 0
    (2, 8),
    (10, 9),
    (9, 0),
    # (0, 1),
    (10, 7),
    (1, 1),
    (6, 10),
    (10, 0),
    (2, 0),
    (11, 6),
    (8, 3),
    (9, 4),
    (2, 2),
    (5, 1),
    # (0, 6),
    (4, 6),
    (2, 4),
    # (0, 7),
    (7, 10),
    (9, 3),
    (4, 3),
    (6, 2),
    (11, 5),
    # (0, 5),
    (7, 1),
    (7, 3),
    (6, 6),
    (3, 7),
    (6, 8),
    # (0, 0),
    (11, 11),
    (7, 8),
    (10, 2),
    (11, 3),
    (7, 9),
    (1, 3),
]

# # NOTE: these are the lowest entropy heads. Ablating just a single one of these breaks the parroting
# # Specifically, these heads were identified from a context-parroting window of Lorenz system (see notebooks/kernel_analogy/head_sharpness.ipynb)
# heads_to_ablate = [
#     (4, 10),
#     # (3, 10),
#     # (4, 1),
#     # (7, 11),
#     # (6, 9),
#     # (3, 9),
#     # (3, 1),
#     # (2, 3),
#     # (5, 6),
#     # (3, 2),
#     # (5, 3),
#     # (3, 5),
#     # (1, 2),
#     # (2, 7),
#     # (6, 4),
#     # (8, 8),
#     # (5, 0),
#     # (7, 0),
#     # (5, 5),
#     # (5, 11),
#     # (1, 11),
#     # (8, 10),
#     # (2, 10),
#     # (8, 0),
#     # (11, 4),
#     # (4, 8),
#     # (8, 1),
#     # (8, 2),
#     # (8, 7),
#     # (4, 5),
#     # (3, 3),
#     # (8, 9),
#     # (3, 4),
#     # (8, 11),
#     # (0, 3),
#     # (6, 7),
# ]


######### Attention Head ablations #########
pipeline.add_head_ablation_hooks(
    heads_to_ablate,
    attention_type="ca",
)
pipeline.add_head_ablation_hooks(
    heads_to_ablate,
    attention_type="sa",
)

######### MLP ablations #########
# pipeline.add_mlp_ablation_hooks(layers_to_ablate, ablation_method="zero")
# pipeline.add_mlp_ablation_hooks([1, 2], ablation_method="zero")

In [ ]:
n_heads_ablated = len(heads_to_ablate)
print(f"n_heads_ablated: {n_heads_ablated}")

In [ ]:
# print number of heads ablated per layer
for layer in list(range(pipeline.num_layers)):
    heads_in_layer = [config[1] for config in heads_to_ablate if config[0] == layer]
    print(f"Layer {layer}: {len(heads_in_layer)} heads")

In [ ]:
layers_without_ca_heads = list(pipeline.ca_head_ablation_handles.keys())
layers_without_sa_heads = list(pipeline.sa_head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

assert layers_without_ca_heads == layers_without_sa_heads, "CA and SA ablations must be identical"
layers_without_heads = layers_without_ca_heads

ablations_summary_str = f"ablate_{n_heads_ablated}_heads"

if layers_without_heads and layers_without_mlps:
    if layers_without_heads != layers_without_mlps:
        # raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
        pass
    ablations_summary_str_title = f"Zero-Ablation: Heads and MLPs in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_mlps_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str_suffix = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str_suffix = ablations_summary_str_title = None

ablations_summary_str += "_" + ablations_summary_str_suffix

print(ablations_summary_str)
print(ablations_summary_str_title)


In [ ]:
save_fname = (
    f"predictions_{system_name_pretty}_term{term}_dim{selected_dim}_start{context_start_time}_sub{subsample_interval}"
    if split_name == "base40"
    else f"predictions_{system_name_pretty}"
)

### Predict and Get Outputs

In [ ]:
rseed = 123
set_seed(rseed)

num_samples = 20
do_sample = True if num_samples > 1 else False

pred = pipeline.predict(  # type: ignore
    context,
    prediction_length=prediction_length,
    num_samples=num_samples,
    limit_prediction_length=False,
)

### Plot Predictions

In [ ]:
# Prepare context and predictions
context_np = context.squeeze().cpu().numpy()
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.squeeze()
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)
print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")

In [ ]:
preds = pred.squeeze()
if preds.ndim == 1:
    preds = preds[None, :]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

# Plot context, ground truth and predictions
ax.plot(context_timesteps[-512:], context_np[-512:], color="black", linewidth=1, label="Context")
ax.plot(
    future_timesteps[: future_vals_np.shape[-1]],
    future_vals_np,
    color="black",
    linewidth=1,
    linestyle="--",
    label="Ground Truth",
)

# ax.set_ylim(min(context_np[-512:].min(), median_pred.min()), max(context_np[-512:].max(), median_pred.max()))
# Calculate y-axis limits, handling potential NaN/Inf values
median_pred = np.median(preds, axis=0)
y_min = min(context_np[-512:].min(), median_pred.min(), future_vals_np.min())
y_max = max(context_np[-512:].max(), median_pred.max(), future_vals_np.max())
y_range = y_max - y_min
y_min = y_min - 0.01 * y_range
y_max = y_max + 0.01 * y_range

# Only set limits if they are finite
if np.isfinite(y_min) and np.isfinite(y_max):
    ax.set_ylim(y_min, y_max)

for i in range(len(preds)):
    ax.plot(future_timesteps, preds[i], color=DEFAULT_COLORS[i], linewidth=0.8, alpha=0.33)
ax.plot(future_timesteps, median_pred, color="tab:blue", linewidth=2, label="Median")

ax.set_xlabel("Timestep", fontweight="bold")
# ax.set_title(system_name, fontweight="bold", fontsize=10)

# Save plot
save_path = os.path.join(plot_save_dir, "zero_ablations", f"{ablations_summary_str}_{save_fname}.pdf")
if ablations_summary_str_title:
    print(ablations_summary_str_title)
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
fig.savefig(save_path, bbox_inches="tight")

In [ ]:
pipeline.remove_all_hooks()
set_seed(rseed)

pred_original = pipeline.predict(  # type: ignore
    context,
    prediction_length=prediction_length,
    num_samples=num_samples,
    limit_prediction_length=False,
)

In [ ]:
preds_original = pred_original.squeeze()
if preds_original.ndim == 1:
    preds_original = preds_original[None, :]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

# Plot context, ground truth and predictions
ax.plot(context_timesteps[-512:], context_np[-512:], color="black", linewidth=1, label="Context")
ax.plot(
    future_timesteps[: future_vals_np.shape[-1]],
    future_vals_np,
    color="black",
    linewidth=1,
    linestyle="--",
    label="Ground Truth",
)

# ax.set_ylim(min(context_np[-512:].min(), median_pred.min()), max(context_np[-512:].max(), median_pred.max()))
# Calculate y-axis limits, handling potential NaN/Inf values
median_pred_original = np.median(preds_original, axis=0)
y_min = min(context_np[-512:].min(), median_pred_original.min(), future_vals_np.min())
y_max = max(context_np[-512:].max(), median_pred_original.max(), future_vals_np.max())
y_range = y_max - y_min
y_min = y_min - 0.01 * y_range
y_max = y_max + 0.01 * y_range

# Only set limits if they are finite
if np.isfinite(y_min) and np.isfinite(y_max):
    ax.set_ylim(y_min, y_max)

for i in range(len(preds_original)):
    ax.plot(future_timesteps, preds_original[i], color=DEFAULT_COLORS[i], linewidth=0.8, alpha=0.33)
ax.plot(future_timesteps, median_pred_original, color="tab:blue", linewidth=2, label="Median")

ax.set_xlabel("Timestep", fontweight="bold")
# ax.set_title(system_name, fontweight="bold", fontsize=10)

# Save plot
save_path = os.path.join(plot_save_dir, "original", f"{save_fname}.pdf")
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
fig.savefig(save_path, bbox_inches="tight")

### Compute Metrics

In [ ]:
med_pred = np.median(preds, axis=0)
med_orig = np.median(preds_original, axis=0)
print(f"med_pred shape: {med_pred.shape}")
print(f"med_orig shape: {med_orig.shape}")

In [ ]:
pred_horizon_lst = [64]
for pred_horizon in pred_horizon_lst:
    print(f"Computing metrics for prediction horizon {pred_horizon}")
    metrics_combined = compute_metrics(med_orig[:pred_horizon], med_pred[:pred_horizon])
    pprint(metrics_combined, width=1, indent=2)